In [5]:
!mkdir 01_raw
!mkdir 02_interim

In [12]:
file_mapping = {
    '01_raw/zone_NE.xlsx': 'NE',
    '01_raw/zone_NW.xlsx': 'NW',
    '01_raw/zone_SE.xlsx': 'SE',
    '01_raw/zone_SW.xlsx': 'SW'
}

In [13]:
file_mapping = {
    '01_raw/zone_NE.xlsx': 'NE',
    '01_raw/zone_NW.xlsx': 'NW',
    '01_raw/zone_SE.xlsx': 'SE',
    '01_raw/zone_SW.xlsx': 'SW'
}

In [15]:
import pandas as pd
import os

# Assicuriamoci che la cartella di destinazione esista
os.makedirs('02_interim', exist_ok=True)

# NOMI CORRETTI dei file effettivamente presenti nella cartella '01_raw/'
file_mapping = {
    '01_raw/Storey1_NE_report_by_device.xlsx': 'NE',
    '01_raw/Storey1_NW_report_by_device.xlsx': 'NW',
    '01_raw/Storey1_SE_report_by_device.xlsx': 'SE',
    '01_raw/Storey1_SW_report_by_device.xlsx': 'SW'
}

csv_totali_creati = 0

# Leggi ogni file Excel ed estrai i suoi fogli
for file_path, zone in file_mapping.items():
    print(f"Sto leggendo il file: {file_path} (Zona: {zone}). Potrebbe volerci qualche minuto...")

    # Legge TUTTI i fogli presenti nel file
    tutti_i_fogli = pd.read_excel(file_path, sheet_name=None)

    for nome_foglio, df in tutti_i_fogli.items():
        load_type = nome_foglio.strip()

        # Aggiungi le colonne (Task A1)
        df['zone'] = zone
        df['load_type'] = load_type

        # Genera il nome del file CSV
        nome_pulito_load = load_type.replace(' ', '_').replace('&', 'and')
        percorso_output = f"02_interim/{zone}_{nome_pulito_load}.csv"

        # Salva il foglio come CSV
        df.to_csv(percorso_output, index=False)
        csv_totali_creati += 1

        print(f"  -> Salvato: {percorso_output} (Righe: {len(df)})")

print(f"\nOperazione conclusa! Totale file creati: {csv_totali_creati} (Dovrebbero essere 18).")

Sto leggendo il file: 01_raw/Storey1_NE_report_by_device.xlsx (Zona: NE). Potrebbe volerci qualche minuto...
  -> Salvato: 02_interim/NE_HVAC.csv (Righe: 86719)
  -> Salvato: 02_interim/NE_Plugs_and_Sockets.csv (Righe: 88826)
Sto leggendo il file: 01_raw/Storey1_NW_report_by_device.xlsx (Zona: NW). Potrebbe volerci qualche minuto...
  -> Salvato: 02_interim/NW_Emerg._Lighting.csv (Righe: 86686)
  -> Salvato: 02_interim/NW_HVAC.csv (Righe: 86580)
  -> Salvato: 02_interim/NW_Hand_Dryer.csv (Righe: 88190)
  -> Salvato: 02_interim/NW_Lighting.csv (Righe: 88355)
  -> Salvato: 02_interim/NW_Plugs_and_Sockets.csv (Righe: 88883)
  -> Salvato: 02_interim/NW_RACK.csv (Righe: 86928)
Sto leggendo il file: 01_raw/Storey1_SE_report_by_device.xlsx (Zona: SE). Potrebbe volerci qualche minuto...
  -> Salvato: 02_interim/SE_Emerg._Lighting.csv (Righe: 63970)
  -> Salvato: 02_interim/SE_HVAC.csv (Righe: 63958)
  -> Salvato: 02_interim/SE_Lighting.csv (Righe: 72551)
  -> Salvato: 02_interim/SE_Plugs_and_S

In [17]:
import pandas as pd
import os
import numpy as np
import glob

# Crea una nuova cartella per i file aggiornati del Task A2
os.makedirs('02_interim_A2', exist_ok=True)

# Trova tutti i 18 file CSV creati prima
csv_files = glob.glob('02_interim/*.csv')

# Dizionario per rinominare le colonne esattamente come richiesto
rename_rules = {
    'Timestamp': 'timestamp',
    'W': 'power_w',
    'Pa': 'apparent_power_va',
    'I': 'current_a',
    'Er': 'active_energy_counter_wh',
    'Ea': 'apparent_energy_counter',
    'Number of Edevices': 'n_devices', # Nome originale dei dispositivi
    'N_Devices': 'n_devices',          # Alternativa per sicurezza
    'Temp': 'temp_c',
    'Hum': 'hum_pct',
    'Lum': 'lum_lux',
    'Pres': 'occupancy'
}

print(f"Inizio elaborazione Task A2 su {len(csv_files)} file...")

for file_path in csv_files:
    # Leggi il file CSV temporaneo
    df = pd.read_csv(file_path)

    # 1. Rinomina le colonne
    df.rename(columns=rename_rules, inplace=True)

    # 2. Assicurati che il Timestamp sia in un formato "data e ora" per Python
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # 3. Ordina per data (dal 2020 al 2025) ed elimina le righe esattamente duplicate
    df = df.sort_values('timestamp')
    df = df.drop_duplicates()

    # 4. Aggiungi la colonna 'period' (il gap di 902 giorni divide il <=2023 dal 2025)
    df['period'] = np.where(df['timestamp'].dt.year <= 2023, 'hourly_2020_2023', 'minute_2025')

    # Salva il file sistemato nella nuova cartella
    nome_file = os.path.basename(file_path)
    nuovo_percorso = os.path.join('02_interim_A2', nome_file)
    df.to_csv(nuovo_percorso, index=False)

    print(f"✔ Aggiornato e salvato: {nome_file} (Righe dopo pulizia doppioni: {len(df)})")

print("\nTask A2 completato! I file sono pronti nella cartella '02_interim_A2'.")

Inizio elaborazione Task A2 su 18 file...
✔ Aggiornato e salvato: NW_Hand_Dryer.csv (Righe dopo pulizia doppioni: 88190)
✔ Aggiornato e salvato: SE_Plugs_and_Sockets.csv (Righe dopo pulizia doppioni: 83126)
✔ Aggiornato e salvato: NW_Lighting.csv (Righe dopo pulizia doppioni: 88355)
✔ Aggiornato e salvato: NW_Emerg._Lighting.csv (Righe dopo pulizia doppioni: 86686)
✔ Aggiornato e salvato: SW_RACK.csv (Righe dopo pulizia doppioni: 13038)
✔ Aggiornato e salvato: NE_Plugs_and_Sockets.csv (Righe dopo pulizia doppioni: 88826)
✔ Aggiornato e salvato: NE_HVAC.csv (Righe dopo pulizia doppioni: 86719)
✔ Aggiornato e salvato: SE_Projector.csv (Righe dopo pulizia doppioni: 61689)
✔ Aggiornato e salvato: SW_Plugs_and_Sockets.csv (Righe dopo pulizia doppioni: 69466)
✔ Aggiornato e salvato: SW_Lighting.csv (Righe dopo pulizia doppioni: 74903)
✔ Aggiornato e salvato: SW_Emerg._Lighting.csv (Righe dopo pulizia doppioni: 65383)
✔ Aggiornato e salvato: NW_RACK.csv (Righe dopo pulizia doppioni: 86928)
✔ 

In [18]:
import pandas as pd
import os
import glob

# Crea la cartella per i file salvati del Task A3
os.makedirs('02_interim_A3', exist_ok=True)

# Prendi i file puliti dal task precedente
csv_files = glob.glob('02_interim_A2/*.csv')

# Definiamo le regole matematiche richieste per l'aggregazione oraria (Task A3)
agg_rules = {
    'power_w': 'mean',                   # Media per la potenza
    'apparent_power_va': 'mean',         # Media per la potenza apparente
    'current_a': 'mean',                 # Media per la corrente
    'temp_c': 'mean',                    # Media per temperatura
    'hum_pct': 'mean',                   # Media per umidità
    'lum_lux': 'mean',                   # Media per luminosità
    'occupancy': 'mean',                 # Media (trasforma i 0/1 in percentuale di occupazione dell'ora)
    'active_energy_counter_wh': 'last',  # ULTIMO valore dell'ora per il contachilometri
    'apparent_energy_counter': 'last',   # ULTIMO valore per il contatore apparente
    'zone': 'first',                     # Mantiene il nome della zona
    'load_type': 'first',                # Mantiene il tipo di carico
    'n_devices': 'first',                # Mantiene il numero di dispositivi
    'period': 'first'                    # Mantiene l'etichetta del periodo
}

print(f"Inizio ricampionamento orario (Task A3) su {len(csv_files)} file...")

for file_path in csv_files:
    df = pd.read_csv(file_path)

    # Assicurati che timestamp sia di tipo datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # Imposta la data come "indice" (obbligatorio per usare la funzione resample)
    df.set_index('timestamp', inplace=True)

    # Filtriamo le regole: applichiamo solo quelle per le colonne che esistono davvero nel file
    current_agg_rules = {col: agg_rules[col] for col in agg_rules if col in df.columns}

    # Esegue il ricampionamento orario ('1h') applicando le regole di media e ultimo valore
    df_hourly = df.resample('1h').agg(current_agg_rules)

    # Il ricampionamento crea righe vuote per il buco di 902 giorni tra il 2023 e il 2025.
    # Il documento dice: "Do not fill gaps here; missing hours stay missing."
    # Quindi eliminiamo le righe artificiali in cui la 'zone' è diventata vuota.
    df_hourly = df_hourly.dropna(subset=['zone'])

    # Riporta il timestamp a essere una normale colonna e non più un indice
    df_hourly.reset_index(inplace=True)

    # Salva il file
    nome_file = os.path.basename(file_path)
    nuovo_percorso = os.path.join('02_interim_A3', nome_file)
    df_hourly.to_csv(nuovo_percorso, index=False)

    print(f"✔ Ricampionato: {nome_file} (Righe orarie finali: {len(df_hourly)})")

print("\nTask A3 completato! Tutti i file sono ora esattamente orari nella cartella '02_interim_A3'.")

Inizio ricampionamento orario (Task A3) su 18 file...
✔ Ricampionato: NW_Hand_Dryer.csv (Righe orarie finali: 26301)
✔ Ricampionato: SE_Plugs_and_Sockets.csv (Righe orarie finali: 25737)
✔ Ricampionato: NW_Lighting.csv (Righe orarie finali: 26301)
✔ Ricampionato: NW_Emerg._Lighting.csv (Righe orarie finali: 24798)
✔ Ricampionato: SW_RACK.csv (Righe orarie finali: 13038)
✔ Ricampionato: NE_Plugs_and_Sockets.csv (Righe orarie finali: 26301)
✔ Ricampionato: NE_HVAC.csv (Righe orarie finali: 26301)
✔ Ricampionato: SE_Projector.csv (Righe orarie finali: 24934)
✔ Ricampionato: SW_Plugs_and_Sockets.csv (Righe orarie finali: 25582)
✔ Ricampionato: SW_Lighting.csv (Righe orarie finali: 25677)
✔ Ricampionato: SW_Emerg._Lighting.csv (Righe orarie finali: 25519)
✔ Ricampionato: NW_RACK.csv (Righe orarie finali: 26300)
✔ Ricampionato: NW_Plugs_and_Sockets.csv (Righe orarie finali: 26301)
✔ Ricampionato: SW_Hand_Dryer.csv (Righe orarie finali: 25511)
✔ Ricampionato: SE_HVAC.csv (Righe orarie finali:

In [19]:
import pandas as pd
import numpy as np
import os
import glob

# Crea la cartella per i file salvati del Task A4
os.makedirs('02_interim_A4', exist_ok=True)

# Prendi i file orari dal task precedente
csv_files = glob.glob('02_interim_A3/*.csv')

print(f"Inizio calcolo energia e flag reset (Task A4) su {len(csv_files)} file...")

for file_path in csv_files:
    df = pd.read_csv(file_path)

    # Ordiniamo per sicurezza cronologicamente, fondamentale prima di fare la differenza
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')

    # Inizializza flag_reset a 0 per tutte le righe
    df['flag_reset'] = 0

    # Controlliamo che il file abbia le colonne di potenza e contatore
    if 'active_energy_counter_wh' in df.columns and 'power_w' in df.columns:

        # 1. Calcola l'energia oraria (differenza rispetto alla riga precedente)
        df['energy_wh'] = df['active_energy_counter_wh'].diff()

        # 2. Calcola il tetto massimo plausibile richiesto dal documento (3x 99° percentile)
        max_plausible = 3 * df['power_w'].quantile(0.99)

        # 3. Individua i reset: valore negativo OPPURE esageratamente alto
        mask_anomalia = (df['energy_wh'] < 0) | (df['energy_wh'] > max_plausible)

        # 4. Applica la regola: energia diventa NaN, flag diventa 1
        df.loc[mask_anomalia, 'energy_wh'] = np.nan
        df.loc[mask_anomalia, 'flag_reset'] = 1

    else:
        # Per sicurezza, se un file non ha il contatore, creiamo la colonna vuota
        df['energy_wh'] = np.nan

    # Salva il file elaborato
    nome_file = os.path.basename(file_path)
    nuovo_percorso = os.path.join('02_interim_A4', nome_file)
    df.to_csv(nuovo_percorso, index=False)

    # Conta quanti reset sono stati bloccati
    tot_reset = df['flag_reset'].sum()
    print(f"✔ Elaborato: {nome_file} (Trovati {tot_reset} reset)")

print("\nTask A4 completato! I file con energia calcolata sono nella cartella '02_interim_A4'.")

Inizio calcolo energia e flag reset (Task A4) su 18 file...
✔ Elaborato: NW_Hand_Dryer.csv (Trovati 13955 reset)
✔ Elaborato: SE_Plugs_and_Sockets.csv (Trovati 4350 reset)
✔ Elaborato: NW_Lighting.csv (Trovati 475 reset)
✔ Elaborato: NW_Emerg._Lighting.csv (Trovati 24325 reset)
✔ Elaborato: SW_RACK.csv (Trovati 1157 reset)
✔ Elaborato: NE_Plugs_and_Sockets.csv (Trovati 406 reset)
✔ Elaborato: NE_HVAC.csv (Trovati 84 reset)
✔ Elaborato: SE_Projector.csv (Trovati 0 reset)
✔ Elaborato: SW_Plugs_and_Sockets.csv (Trovati 5114 reset)
✔ Elaborato: SW_Lighting.csv (Trovati 13376 reset)
✔ Elaborato: SW_Emerg._Lighting.csv (Trovati 4956 reset)
✔ Elaborato: NW_RACK.csv (Trovati 49 reset)
✔ Elaborato: NW_Plugs_and_Sockets.csv (Trovati 490 reset)
✔ Elaborato: SW_Hand_Dryer.csv (Trovati 3621 reset)
✔ Elaborato: SE_HVAC.csv (Trovati 2866 reset)
✔ Elaborato: SE_Emerg._Lighting.csv (Trovati 2864 reset)
✔ Elaborato: SE_Lighting.csv (Trovati 4166 reset)
✔ Elaborato: NW_HVAC.csv (Trovati 169 reset)

Task 

In [20]:
import pandas as pd
import os
import glob

# 1. Crea la cartella finale richiesta dal progetto
os.makedirs('03_processed', exist_ok=True)

# Prendi i 18 file orari dal Task A4
csv_files = glob.glob('02_interim_A4/*.csv')

print(f"Inizio unione finale (Task A5) di {len(csv_files)} file...")

# Lista per contenere tutti i dataframe
lista_df = []

for file_path in csv_files:
    df = pd.read_csv(file_path)
    lista_df.append(df)

# 2. Impila tutti i file uno sopra l'altro (concat)
df_all = pd.concat(lista_df, ignore_index=True)
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'])

# 3. Estrai i dati ambientali a livello di ZONA per evitare conflitti
colonne_ambientali = ['timestamp', 'zone', 'temp_c', 'hum_pct', 'lum_lux', 'occupancy']
# Prendiamo solo le colonne ambientali, togliamo i duplicati e teniamo i valori validi
df_env = df_all[colonne_ambientali].dropna(subset=['temp_c']).drop_duplicates(subset=['timestamp', 'zone'])

# Rimuovi le vecchie colonne ambientali dal dataframe principale
df_energia = df_all.drop(columns=['temp_c', 'hum_pct', 'lum_lux', 'occupancy'])

# 4. Riattacca i dati ambientali una volta sola per zona (merge)
df_finale = pd.merge(df_energia, df_env, on=['timestamp', 'zone'], how='left')

# 5. Aggiungi i flag mancanti inizializzati a 0 (li userà la Sottosquadra B)
df_finale['flag_missing'] = 0
df_finale['flag_negative'] = 0
df_finale['flag_outlier'] = 0

# Se flag_reset ha dei NaN (dovuti alle unioni), mettili a 0
if 'flag_reset' in df_finale.columns:
    df_finale['flag_reset'] = df_finale['flag_reset'].fillna(0).astype(int)

# 6. Riordina le colonne esattamente come richiesto nel layout target del progetto
colonne_target = [
    'timestamp', 'zone', 'load_type', 'n_devices',
    'power_w', 'apparent_power_va', 'current_a', 'energy_wh',
    'temp_c', 'hum_pct', 'lum_lux', 'occupancy',
    'period', 'flag_missing', 'flag_reset', 'flag_negative', 'flag_outlier'
]

# Mantieni solo le colonne richieste (evitando errori se qualcuna manca)
colonne_presenti = [col for col in colonne_target if col in df_finale.columns]
df_finale = df_finale[colonne_presenti]

# Ordina cronologicamente e per zona
df_finale = df_finale.sort_values(['timestamp', 'zone', 'load_type'])

# 7. Salva il capolavoro nella cartella finale
percorso_finale = '03_processed/cedint_f1_hourly.csv'
df_finale.to_csv(percorso_finale, index=False)

print(f"\n🎉 Task A5 completato con successo!")
print(f"Il file finale '{percorso_finale}' è pronto.")
print(f"Dimensioni del dataset finale: {df_finale.shape[0]} righe e {df_finale.shape[1]} colonne.")

Inizio unione finale (Task A5) di 18 file...

🎉 Task A5 completato con successo!
Il file finale '03_processed/cedint_f1_hourly.csv' è pronto.
Dimensioni del dataset finale: 450482 righe e 17 colonne.
